In [1]:
import numpy as np
from numba import njit
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigs
import multiprocessing as mp
import os
import time

# --- 1. 加载真值 (黎曼零点权威数据) ---
try:
    TRUE_GAMMAS = np.load("riemann_10k_true.npy")
    print(f">>> 成功加载权威真值，共 {len(TRUE_GAMMAS)} 个零点。")
except:
    print("!!! 错误：未找到 riemann_10k_true.npy")
    exit()

# --- 2. 动力学内核：基于 V6 测绘参数的冷却引擎 ---
@njit(fastmath=True, nogil=True)
def harvesting_kernel_v6(u_c, k_static, steps, n_bins):
    """
    k_static: 由导航公式计算出的该段最优 k
    """
    x = 0.5
    counts = np.zeros((n_bins, n_bins), dtype=np.float64)
    last_bin = 0
    warmup = 2000000 # 200万步热启动确保平稳
    
    for i in range(steps + warmup):
        # 微观衰减项 (ln^2)
        ln_term = np.log(i + 100)
        mu_eff = u_c + k_static / (ln_term**2)
        
        x = 1 - mu_eff * x**2
        
        # 数值截断保护
        if x > 1.0: x = 0.999
        elif x < -1.0: x = -0.999
            
        if i > warmup:
            current_bin = int((x + 1.0) / 2.0 * (n_bins - 1))
            if 0 <= current_bin < n_bins:
                counts[last_bin, current_bin] += 1
                last_bin = current_bin
    return counts

# --- 3. 狙击手 Worker：精准提取正能量相位 ---
def sniper_worker_v6(task):
    seg_idx, start_n, end_n, k_opt = task
    U_C = 1.543689012
    N_BINS = 20000    # 高分辨率
    STEPS = 10**8     # 1亿步确保统计稳定性
    
    SAVE_DIR = "riemann_10k_harvest_pure_V6"
    
    try:
        t0 = time.time()
        # 运行动力学演化
        counts = harvesting_kernel_v6(U_C, k_opt, STEPS, N_BINS)
        
        # 稀疏化与归一化
        P = csr_matrix(counts)
        row_sums = np.array(P.sum(axis=1)).flatten()
        row_sums[row_sums == 0] = 1.0
        P.data /= row_sums[P.indices]
        
        # 提取特征值：索要量翻倍以过滤正能量态
        num_targets = end_n - start_n
        calc_k = num_targets * 2 + 100 
        
        v0 = np.ones(N_BINS)
        vals, _ = eigs(P, k=calc_k, which='LM', tol=1e-4, v0=v0)
        
        # 🌟 核心过滤器：只取正能量态 (Imag > 0) 🌟
        # 过滤掉 abs < 0.4 的噪声和虚部为负的影子态
        pure_vals = vals[(np.abs(vals) > 0.4) & (vals.imag > 1e-7)]
        phases = np.sort(np.angle(pure_vals))
        
        # 现场误差校验
        if len(phases) >= num_targets:
            sim_segment = phases[:num_targets]
            true_segment = TRUE_GAMMAS[start_n:end_n]
            # 对齐第一个点
            scale = true_segment[0] / sim_segment[0]
            avg_err = np.mean(np.abs(sim_segment * scale - true_segment))
        else:
            avg_err = 999.9

        # 保存数据
        if not os.path.exists(SAVE_DIR):
            os.makedirs(SAVE_DIR, exist_ok=True)
        filename = os.path.join(SAVE_DIR, f"seg_{seg_idx}_n_{start_n}_k_{k_opt:.4f}.npy")
        np.save(filename, phases)
        
        return f"DONE | Seg {seg_idx:3} [n={start_n:5}] | k={k_opt:.4f} | Err={avg_err:.4f} | {time.time()-t0:.1f}s"
    except Exception as e:
        return f"FAIL | Seg {seg_idx:3} | {str(e)}"

# --- 4. 战役调度中心 ---
if __name__ == "__main__":
    SAVE_DIR = "riemann_10k_harvest_pure_V6"
    if not os.path.exists(SAVE_DIR):
        os.makedirs(SAVE_DIR)
        
    print(f"="*60)
    print(f"🚀 启动‘万点长征’最终收割 (V6 测绘驱动版)")
    print(f">>> 物理导航公式: k(n) = 5.0969 + 2.4492 / ln(n+100)")
    print(f"="*60)
    
    # 🌟 导航公式参数锁定
    K0 = 5.0969
    BETA = 2.4492
    
    tasks = []
    segment_size = 100
    for i in range(0, 10000, segment_size):
        start_n = i
        end_n = i + segment_size
        mid_n = (start_n + end_n) / 2
        
        # 实时计算该段的最佳 k 值
        k_opt = K0 + BETA / np.log(mid_n + 100)
        tasks.append((i//segment_size, start_n, end_n, k_opt))

    # 480G 内存，支持 100 并发稳定运行
    BATCH_SIZE = 100
    start_total = time.time()
    
    with mp.Pool(processes=BATCH_SIZE) as pool:
        results = pool.map(sniper_worker_v6, tasks)
        
    for res in results:
        print(f"  {res}")

    print(f"\n" + "="*60)
    print(f"✅ 万点正能量数据收割完毕！")
    print(f">>> 总耗时: {(time.time()-start_total)/60:.2f} 分钟")
    print(f">>> 下一步：运行残差可视化脚本，见证‘对数弯钩’被抹平！")
    print(f"="*60)

>>> 成功加载权威真值，共 10000 个零点。
🚀 启动‘万点长征’最终收割 (V6 测绘驱动版)
>>> 物理导航公式: k(n) = 5.0969 + 2.4492 / ln(n+100)
  DONE | Seg   0 [n=    0] | k=5.5857 | Err=99.0977 | 624.9s
  DONE | Seg   1 [n=  100] | k=5.5405 | Err=10377.5334 | 748.5s
  DONE | Seg   2 [n=  200] | k=5.5150 | Err=22654.8760 | 708.9s
  DONE | Seg   3 [n=  300] | k=5.4978 | Err=8199.1343 | 749.1s
  DONE | Seg   4 [n=  400] | k=5.4851 | Err=27111.0747 | 721.0s
  DONE | Seg   5 [n=  500] | k=5.4750 | Err=16206.3964 | 743.2s
  DONE | Seg   6 [n=  600] | k=5.4669 | Err=58380.3723 | 740.7s
  DONE | Seg   7 [n=  700] | k=5.4600 | Err=63868.2691 | 698.4s
  DONE | Seg   8 [n=  800] | k=5.4541 | Err=27772.1883 | 720.6s
  DONE | Seg   9 [n=  900] | k=5.4490 | Err=277073.2255 | 751.6s
  DONE | Seg  10 [n= 1000] | k=5.4444 | Err=77929.4160 | 739.6s
  DONE | Seg  11 [n= 1100] | k=5.4404 | Err=53098.0583 | 741.1s
  DONE | Seg  12 [n= 1200] | k=5.4367 | Err=80283.7099 | 742.3s
  DONE | Seg  13 [n= 1300] | k=5.4334 | Err=72610.4690 | 657.3s
  DONE |